<a href="https://colab.research.google.com/github/vvelvadapu9/DemoAIProj/blob/Dev/Autoencoder_Model_Text_Reconstrcution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Text Reconstruction Using Auto-Encoder**

In [29]:
# Import required libraries
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [31]:
# Sample corpus
sentences = [
    "hello how are you",
    "good morning",
    "what is your name",
    "have a nice day",
    "see you soon",
    "thank you very much",
    "how can I help you",
    "nice to meet you"
]

In [66]:
corpus=open("corpus.txt","r")
sentences = corpus.readlines()
print(sentences)
print(type(sentences))
print(len(sentences))

['Artificial Intelligence (AI) empowers machines to mimic human cognition, learning, and problem-solving, transforming industries from healthcare to entertainment through tasks like voice recognition and complex data analysis. \n', 'It automates processes, enhances decision-making, and provides 24/7 efficiency, exemplified by smart assistants and recommendation engines. While boosting productivity and innovation, AI also presents ethical questions regarding job displacement and bias, requiring careful development. \n', 'Ultimately, AI offers powerful tools for progress, but its responsible integration is crucial for shaping a beneficial future for humanity. ']
<class 'list'>
3


In [67]:
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1
max_len = max(len(s.split()) for s in sentences)

sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=max_len, padding='post')

In [68]:
# LSTM Autoencoder
embedding_dim = 50
latent_dim = 64

# Encoder
inputs = Input(shape=(max_len,))
embed = Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)(inputs)
encoded = LSTM(latent_dim)(embed)

# Decoder
decoded = RepeatVector(max_len)(encoded)
decoded = LSTM(latent_dim, return_sequences=True)(decoded)
outputs = TimeDistributed(Dense(vocab_size, activation='softmax'))(decoded)

autoencoder = Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='sparse_categorical_crossentropy')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [69]:
# Prepare target output (same as input, but reshaped for sparse labels)
targets = np.expand_dims(padded, -1)

In [70]:
# Train
autoencoder.fit(padded, targets, epochs=300, verbose=0)

In [71]:
# Predict and decode
preds = autoencoder.predict(padded)
decoded_sentences = []
for sentence in preds:
    words = [tokenizer.index_word.get(np.argmax(vec), '') for vec in sentence]
    decoded_sentences.append(' '.join(words).strip())

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 935ms/step


In [72]:
# Display results
#print(corpus)
for i, original in enumerate(sentences):
    print(f"Original: {original}")
    print(f"Reconstructed: {decoded_sentences[i]}\n")

Original: Artificial Intelligence (AI) empowers machines to mimic human cognition, learning, and problem-solving, transforming industries from healthcare to entertainment through tasks like voice recognition and complex data analysis. 

Reconstructed: artificial intelligence ai empowers machines to mimic human cognition learning and problem solving transforming industries from healthcare to entertainment through tasks like voice recognition recognition complex data

Original: It automates processes, enhances decision-making, and provides 24/7 efficiency, exemplified by smart assistants and recommendation engines. While boosting productivity and innovation, AI also presents ethical questions regarding job displacement and bias, requiring careful development. 

Reconstructed: processes enhances decision making and provides 24 efficiency efficiency exemplified by smart assistants and and while while productivity and and and ai presents presents questions questions regarding displacement c

# **Sequential Model**

In [13]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample corpus
sentences = [
    "hello how are you",
    "good morning",
    "what is your name",
    "have a nice day",
    "see you soon",
    "thank you very much",
    "how can I help you",
    "nice to meet you"
]
'''
with open("corpus.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

print(corpus)
'''
# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1
max_len = max(len(s.split()) for s in sentences)

sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=max_len, padding='post')

# Prepare target output (same as input, reshaped for sparse labels)
targets = np.expand_dims(padded, -1)

# Sequential model
embedding_dim = 50
latent_dim = 64

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    LSTM(latent_dim),
    RepeatVector(max_len),
    LSTM(latent_dim, return_sequences=True),
    TimeDistributed(Dense(vocab_size, activation='softmax'))
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.summary()

# Train
model.fit(padded, targets, epochs=300, verbose=0)

# Predict and decode
preds = model.predict(padded)
decoded_sentences = []
for sentence in preds:
    words = [tokenizer.index_word.get(np.argmax(vec), '') for vec in sentence]
    decoded_sentences.append(' '.join(words).strip())

# Display results
for i, original in enumerate(sentences):
    print(f"Original: {original}")
    print(f"Reconstructed: {decoded_sentences[i]}\n")

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_10 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_5 (RepeatVector)  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_5              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 396ms/step
Original: hello how are you
Reconstructed: hello how are you

Original: good morning
Reconstructed: good morning

Original: what is your name
Reconstructed: what is your name

Original: have a nice day
Reconstructed: have a nice day

Original: see you soon
Reconstructed: see you soon

Original: thank you very much
Reconstructed: thank you very much

Original: how can I help you
Reconstructed: how can i help you

Original: nice to meet you
Reconstructed: nice to meet you



# **With User Input**

In [10]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, RepeatVector, TimeDistributed, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Sample corpus for training
sentences = [
    "hello how are you",
    "good morning",
    "what is your name",
    "have a nice day",
    "see you soon",
    "thank you very much",
    "how can I help you",
    "nice to meet you"
]

# Tokenization
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1
max_len = max(len(s.split()) for s in sentences)

sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, maxlen=max_len, padding='post')
targets = np.expand_dims(padded, -1)

# Build LSTM Autoencoder
embedding_dim = 50
latent_dim = 64

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    LSTM(latent_dim),
    RepeatVector(max_len),
    LSTM(latent_dim, return_sequences=True),
    TimeDistributed(Dense(vocab_size, activation='softmax'))
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(padded, targets, epochs=300, verbose=0)

# Real-time reconstruction function
def reconstruct_text(input_text):
    seq = tokenizer.texts_to_sequences([input_text])
    padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
    pred = model.predict(padded_seq)
    words = [tokenizer.index_word.get(np.argmax(vec), '') for vec in pred[0]]
    return ' '.join(words).strip()


while True:
    user_input = input("Enter a sentence (or 'exit'): ")
    if user_input.lower() == 'exit':
        break
    reconstructed = reconstruct_text(user_input)
    print(f"Reconstructed: {reconstructed}\n")

Enter a sentence (or 'exit'): how are you
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 379ms/step
Reconstructed: hello how are you

Enter a sentence (or 'exit'): where do you live
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Reconstructed: see you soon

Enter a sentence (or 'exit'): i'll come home
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
Reconstructed: good morning



KeyboardInterrupt: Interrupted by user